In [1]:
from main import run_bot
from demo import run_demo
import config

In [2]:
from config import MIN_R2, OUTPUTS_DIR, create_run_dirs
from broker.connection import connect_ib
from broker.data import fetch_prices, fetch_prices_free
from broker.orders import calculate_position_size, execute_order, get_portfolio_value
from analysis.universe import fetch_company_metadata, get_sp500_tickers
from analysis.correlations import compute_correlations, get_top_correlated_pairs, get_top_inverse_pairs
from analysis.model import predict_price
from analysis.signals import generate_signals
from reporting.charts import (plot_correlation_matrix, plot_market_cap_bars,
                               plot_prediction_analysis, plot_price_series)
from reporting.report import print_report, save_signals_csv

In [3]:
n_tickers = None    # int = top N by market cap | None = full S&P 500 | 'FALLBACK_TICKERS' = hardcoded top-20
mode = 'paper'      # demo | paper | live | signals
execute_trades=True

In [4]:
run_dir, gen_dir, corr_dir = create_run_dirs()

print("\nFetching S&P 500 universe...")
tickers, market_caps = get_sp500_tickers(n=n_tickers)

prices_df = fetch_prices_free(tickers)
if prices_df.empty or len(prices_df.columns) < 5:
    print("✗ Insufficient data. Aborting.")
    return

print("\nCalculating correlations...")
corr_matrix, returns = compute_correlations(prices_df)
top_pairs     = get_top_correlated_pairs(corr_matrix, top_n=10)
inverse_pairs = get_top_inverse_pairs(corr_matrix, top_n=10)

signals_df = generate_signals(prices_df, returns, corr_matrix)

# Enrich signals with company name, sector, founded year, market cap (B)
company_meta = fetch_company_metadata(list(prices_df.columns), market_caps)
signals_df = signals_df.merge(
    company_meta.reset_index().rename(columns={'index': 'ticker'}),
    on='ticker', how='left'
)
# Reorder columns so metadata appears right after ticker
meta_cols = ['company_name', 'sector', 'founded', 'market_cap_B']
other_cols = [c for c in signals_df.columns if c not in ['ticker'] + meta_cols]
signals_df = signals_df[['ticker'] + meta_cols + other_cols]

print_report(signals_df, top_pairs, inverse_pairs)
save_signals_csv(signals_df, run_dir / 'signals.csv')


Fetching S&P 500 universe...
  ✓ 503 tickers fetched from Wikipedia
  Sorting 503 tickers by market cap via yfinance (this takes ~30s)...

Fetching market caps for 503 tickers via yfinance...
  ✓ Market caps retrieved: 503/503
  → Using all 503 S&P 500 tickers


In [9]:
save_plots = 1
if save_plots:
    plot_correlation_matrix(corr_matrix,
                            save_path=corr_dir / 'correlation_matrix.png')

    # General/ — price series highlighted by market cap
    plot_price_series(prices_df, tickers, top_n=15, label='market cap',
                      save_path=gen_dir / 'price_series_market-cap.png')

    # General/ — price series highlighted by highest absolute stock price
    tickers_by_price = sorted(
        prices_df.columns.tolist(),
        key=lambda t: prices_df[t].iloc[-1],
        reverse=True
    )
    plot_price_series(prices_df, tickers_by_price, top_n=15, label='stock price',
                      save_path=gen_dir / 'price_series_stock-price-absolute.png')

    # General/ — price series highlighted by highest normalized return (best performers)
    tickers_by_norm = sorted(
        prices_df.columns.tolist(),
        key=lambda t: prices_df[t].iloc[-1] / prices_df[t].iloc[0],
        reverse=True
    )
    plot_price_series(prices_df, tickers_by_norm, top_n=15, label='normalized return',
                      save_path=gen_dir / 'price_series_normalized-return.png')

    # General/ — bar chart: top 15 vs bottom 15 by market cap
    plot_market_cap_bars(prices_df, tickers, market_caps=market_caps, top_n=15,
                         save_path=gen_dir / 'market_cap_bars.png')

    # Correlation_method/ — per-ticker prediction analysis
    top_signals_n = 15
    top_signals = signals_df.head(top_signals_n)
    if not top_signals.empty:
        print("\nGenerating analysis charts...")
    for _, row in top_signals.iterrows():
        ticker = row['ticker']
        pred_ret, r2, top5, corr_signs, y_actual, y_pred = predict_price(
            ticker, returns, corr_matrix
        )
        if y_actual is not None:
            plot_prediction_analysis(
                ticker, returns, prices_df, top5, corr_signs,
                y_actual, y_pred,
                save_path=corr_dir / f'analysis_{ticker}.png'
            )

  Correlation matrix saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/Correlation_method/correlation_matrix.png


In [15]:
execute_trades = 0
if execute_trades:
    ib = connect_ib()
    try:
        portfolio_value = get_portfolio_value(ib)
        print(f"\nPlacing orders (portfolio: ${portfolio_value:,.0f})...")
        actionable = signals_df[signals_df['signal'].isin(['BUY', 'SELL'])]
        for _, row in actionable.iterrows():
            if row['model_r2'] < MIN_R2:
                continue
            strength = min(1.0, row['model_r2'])
            qty = calculate_position_size(portfolio_value, row['current_price'], strength)
            execute_order(ib, row['ticker'], row['signal'], qty)
    finally:
        ib.disconnect()
        print("\n✓ Disconnected from Interactive Brokers.")
else:
    print("\n  ℹ Simulation mode — no orders placed.")
    print("    To execute on paper trading: run_bot(execute_trades=True)")


  ℹ Simulation mode — no orders placed.
    To execute on paper trading: run_bot(execute_trades=True)
